# Phikon-v2: Stain Normalisation and Retrieval Evaluation

This notebook runs the complete pipeline for Phikon-v2:
1. Download and tile PLISM WSIs
2. Extract features under three normalisation conditions
3. Evaluate paired top-1 nearest-neighbour retrieval

## 1. Setup

In [ ]:
%cd /kaggle/working

!pip install -q --no-deps torch-staintools
!pip install -q kornia

import os, shutil, glob

# Copy project scripts from dataset
found = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "plism_loader.py" in files:
        found = root
        break

for pattern in ("*.py", "*.parquet", "*.csv"):
    for source in glob.glob(os.path.join(found, pattern)):
        shutil.copy(source, "/kaggle/working")

print("Setup complete. Project files:", found)

## 2. Download and Tile PLISM

In [ ]:
%%time
%cd /kaggle/working

!python plism_loader.py fetch \
    --tiles-per-slide 400 \
    --out-dir tiles

In [ ]:
!find /kaggle/working/tiles -name "*.h5" | wc -l
!du -sh /kaggle/working/tiles

## 3. Feature Extraction

Extract Phikon-v2 embeddings under three normalisation conditions.

In [ ]:
%%time
# No normalisation
!python extract_features.py \
    --tiles-dir /kaggle/working/tiles \
    --model phikon_v2 \
    --normalise none \
    --out-dir /kaggle/working/features/phikon_v2_none \
    --batch-size 32

In [ ]:
%%time
# Reinhard normalisation
!python extract_features.py \
    --tiles-dir /kaggle/working/tiles \
    --model phikon_v2 \
    --normalise reinhard \
    --out-dir /kaggle/working/features/phikon_v2_reinhard \
    --batch-size 32

In [ ]:
%%time
# Macenko normalisation
!python extract_features.py \
    --tiles-dir /kaggle/working/tiles \
    --model phikon_v2 \
    --normalise macenko \
    --out-dir /kaggle/working/features/phikon_v2_macenko \
    --batch-size 32

## 4. Retrieval Evaluation

In [ ]:
import glob
import itertools
import h5py
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F


def run_retrieval(feature_dir, run_name, plot_title):
    """Paired top-1 nearest-neighbour retrieval evaluation."""

    files = sorted(glob.glob(f"{feature_dir}/*.h5"))
    assert len(files) == 91, f"Expected 91 files, found {len(files)}"

    device = "cuda" if torch.cuda.is_available() else "cpu"
    embeddings = {}
    reference_tile_ids = None

    for file in files:
        with h5py.File(file, "r") as h:
            stainer = str(h.attrs["stainer"])
            scanner = str(h.attrs["scanner"])
            tile_ids = h["tile_id"][:]

            if reference_tile_ids is None:
                reference_tile_ids = tile_ids
            else:
                assert np.array_equal(reference_tile_ids, tile_ids)

            x = torch.from_numpy(h["features"][:]).to(device)
            embeddings[(stainer, scanner)] = F.normalize(x, dim=1)

    stainers = sorted({k[0] for k in embeddings})
    scanners = sorted({k[1] for k in embeddings})
    truth = torch.arange(len(reference_tile_ids), device=device)

    def score(a, b):
        sim = a @ b.T
        fwd = (sim.argmax(dim=1) == truth).float().mean().item()
        rev = (sim.argmax(dim=0) == truth).float().mean().item()
        return fwd, rev, (fwd + rev) / 2

    results = []
    with torch.inference_mode():
        for stainer in stainers:
            for a, b in itertools.combinations(scanners, 2):
                f, r, m = score(embeddings[(stainer, a)], embeddings[(stainer, b)])
                results.append({"axis": "cross-scanner", "fixed": stainer,
                                "a": a, "b": b, "top1_mean": m})
        for scanner in scanners:
            for a, b in itertools.combinations(stainers, 2):
                f, r, m = score(embeddings[(a, scanner)], embeddings[(b, scanner)])
                results.append({"axis": "cross-staining", "fixed": scanner,
                                "a": a, "b": b, "top1_mean": m})

    df = pd.DataFrame(results)
    df["top1_percent"] = df["top1_mean"] * 100

    summary = df.groupby("axis")["top1_percent"].agg(["count", "mean", "std"])
    print(f"\n{plot_title}")
    print(summary.round(2))

    df.to_csv(f"/kaggle/working/retrieval_{run_name}.csv", index=False)
    return df, summary

In [ ]:
# No normalisation
none_results, none_summary = run_retrieval(
    "/kaggle/working/features/phikon_v2_none",
    "phikon_v2_none",
    "Phikon-v2: No normalisation"
)

In [ ]:
# Reinhard normalisation
reinhard_results, reinhard_summary = run_retrieval(
    "/kaggle/working/features/phikon_v2_reinhard",
    "phikon_v2_reinhard",
    "Phikon-v2: Reinhard normalisation"
)

In [ ]:
# Macenko normalisation
macenko_results, macenko_summary = run_retrieval(
    "/kaggle/working/features/phikon_v2_macenko",
    "phikon_v2_macenko",
    "Phikon-v2: Macenko normalisation"
)